# 03 · De imputaciones a fases

Las imputaciones del MES se registran por sesión de trabajo, pero la unidad de análisis del proyecto es la fase.
Este notebook se queda con las imputaciones de las máquinas seleccionadas, resuelve las fases imputadas en varias máquinas, agrega las imputaciones en una fila por fase y descarta las fases con el MES sin cerrar.

Se parte de la salida que dio el notebook `01`.

- **Entrada:** `data/processed/df_imputations_limpieza.parquet` y `data/processed/maquina_grupo.parquet`
- **Salida:** `data/processed/df_fases_MES.parquet` y `results/tables/embudo_03.json`

In [36]:
import sys
from pathlib import Path

# Raíz del repositorio (la carpeta que contiene src/), para poder importar src
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(RAIZ)) # asignar la ruta

import pandas as pd

from src.embudo import cargar_embudo, guardar_embudo, tabla_embudo
from src.rutas import PROCESSED

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

embudo = {}  # imputaciones y fases que quedan tras cada paso

## 1. Imputaciones en las máquinas seleccionadas

Todas las imputaciones que no pertenecen a las máquinas seleccionadas en el notebook 02 (`maquina_grupo`)
quedan fuera del proyecto. El código de máquina de `maquina_grupo` (`CodigoMaquina`) es el mismo que el de las
imputaciones (`smNodoId`), lo que permite relacionar ambas tablas directamente.

In [37]:
df_imputations_limpieza = pd.read_parquet(PROCESSED / "df_imputations_limpieza.parquet")
maquina_grupo = pd.read_parquet(PROCESSED / "maquina_grupo.parquet")
print(f"Máquinas: {len(maquina_grupo)} | grupos de máquinas: {maquina_grupo['CodigoGrupo'].nunique()}")

maquinas_validas = maquina_grupo["CodigoMaquina"].unique()
df_imput_productivas = df_imputations_limpieza[df_imputations_limpieza["smNodoId"].isin(maquinas_validas)]

fuera = len(df_imputations_limpieza) - len(df_imput_productivas)
print(f"Imputaciones antes: {len(df_imputations_limpieza)}")
print(f"Imputaciones después: {len(df_imput_productivas)}")
print(f"Eliminadas (imputaciones fuera de la población de estudio): {fuera} ({fuera / len(df_imputations_limpieza):.2%})")

Máquinas: 32 | grupos de máquinas: 9
Imputaciones antes: 140485
Imputaciones después: 59574
Eliminadas (imputaciones fuera de la población de estudio): 80911 (57.59%)


## 2. Incidencias

Hay imputaciones en máquinas que corresponden a incidencias. No son tiempo productivo, así que se descartan.

In [38]:
df_incidencias = df_imput_productivas[df_imput_productivas["smIncidenciaId"].notnull()]
print(f"Imputaciones de incidencias: {len(df_incidencias)}")

df_imput_productivas = df_imput_productivas[df_imput_productivas["smIncidenciaId"].isnull()]
embudo["Imputaciones en máquinas productivas"] = len(df_imput_productivas)
print(f"Imputaciones restantes: {len(df_imput_productivas)}")

Imputaciones de incidencias: 4
Imputaciones restantes: 59570


## 3. Fases imputadas en varias máquinas

Interesa saber si una misma fase se ha ejecutado en distintas máquinas, es decir, si todas las imputaciones de una
fase se han hecho en la misma máquina. Cada imputación indica en qué nodo se ha realizado.

In [39]:
nodos_por_fase = df_imput_productivas.groupby("SmFaseUid")["smNodoId"].nunique()
embudo["Fases en máquinas productivas"] = len(nodos_por_fase)

print(f"Fases productivas: {len(nodos_por_fase)}\n")
print("Número de máquinas distintas por fase:")
print(nodos_por_fase.value_counts().to_string())

fases_multi = nodos_por_fase[nodos_por_fase > 1].index
n_multi = len(fases_multi)
tiempo_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]["MinutosRealNbr"].sum()
tiempo_total = df_imput_productivas["MinutosRealNbr"].sum()

print(f"\nFases multi-nodo: {n_multi} ({n_multi / len(nodos_por_fase):.1%})")
print(f"Tiempo real en fases multi-nodo: {tiempo_multi / tiempo_total:.1%} del total")

Fases productivas: 19149

Número de máquinas distintas por fase:
smNodoId
1    18250
2      860
3       37
4        2

Fases multi-nodo: 899 (4.7%)
Tiempo real en fases multi-nodo: 15.6% del total


Hay pocas fases multi-nodo (4,7 %), pero con un peso desproporcionado en el tiempo real total (15,6 %): no son
ruido marginal, sino fases más "pesadas" que la media. Según el equipo de producción tiene sentido: son fases
largas o complejas, y precisamente por eso se repartieron entre varias máquinas o sesiones.

Lo importante es saber si en estas fases hay una máquina que concentre casi todo el tiempo.

In [40]:
imput_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]
t = imput_multi.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()
fraccion_dominante_multi = t.groupby("SmFaseUid").max() / t.groupby("SmFaseUid").sum()

print(f"Fases multi-nodo donde la máquina dominante concentra >80 % del tiempo: "
      f"{(fraccion_dominante_multi > 0.8).mean():.1%}")

Fases multi-nodo donde la máquina dominante concentra >80 % del tiempo: 33.5%


En el 33,5 % de las fases multi-nodo la máquina dominante concentra más del 80 % del tiempo. En el 66,5 % restante
el tiempo está genuinamente repartido: la "máquina dominante" no domina tanto y convive con otras máquinas con peso
real.

Se trabaja con las fases en las que la máquina dominante es fácilmente identificable (más del 80 % del tiempo) y
se prescinde del resto. Las fases mono-nodo cumplen el criterio automáticamente (su máquina tiene el 100 %).

In [41]:
# Tiempo por (fase, máquina) y fracción de la máquina dominante en cada fase
tiempo_fase_nodo = df_imput_productivas.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()
dominante = tiempo_fase_nodo.groupby("SmFaseUid").max()
total = tiempo_fase_nodo.groupby("SmFaseUid").sum()
fraccion_dominante = dominante / total

fases_validas = fraccion_dominante[fraccion_dominante > 0.8].index
embudo["Fases válidas (dominante >80%)"] = len(fases_validas)

print(f"Fases antes: {len(fraccion_dominante)}")
print(f"Fases válidas (dominante >80 %): {len(fases_validas)}")
print(f"Fases descartadas: {len(fraccion_dominante) - len(fases_validas)}")
print(f"  de ellas, con tiempo total 0 (la fracción no se puede calcular): {fraccion_dominante.isna().sum()}")

Fases antes: 19149
Fases válidas (dominante >80 %): 18548
Fases descartadas: 601
  de ellas, con tiempo total 0 (la fracción no se puede calcular): 3


Se descartan 601 fases: las 598 multi-nodo con el tiempo repartido y 3 fases mono-nodo cuyo tiempo real total es 0,
para las que la fracción de la máquina dominante no se puede calcular (0/0).

## 4. Tabla final: una fila por fase

**Agregación de imputaciones a nivel de fase**

Los datos de ejecución procedentes del MES se registran a nivel de imputación, de modo que una misma fase de fabricación puede acumular múltiples imputaciones correspondientes a las distintas sesiones de trabajo, operarios o intervalos temporales en que se ejecutó. Sin embargo, la unidad de análisis de este estudio es la fase, no la imputación individual. Por ello, resulta necesario agregar las imputaciones de cada fase en un único registro que la represente de forma completa.
     
Esta agregación responde a dos motivos fundamentales. En primer lugar, la variable objetivo del estudio (el ratio entre el tiempo real y el tiempo teórico) exige que ambos tiempos estén definidos a un mismo nivel. El tiempo teórico se establece en el ERP a nivel de fase, por lo que el tiempo real debe consolidarse igualmente a nivel de
fase para que la comparación entre ambos sea coherente; de lo contrario, se dispondría de múltiples tiempos reales enfrentados a un único tiempo teórico, imposibilitando el cálculo del ratio.
        
En segundo lugar, el modelo que se pretende desarrollar opera sobre la unidad de asignación fase-máquina. Cada observación proporcionada al modelo debe corresponder a una decisión de asignación única. Mantener las imputaciones sin agregar provocaría que una misma fase apareciera representada en varias filas, otorgándole un peso artificialmente elevado en el entrenamiento y distorsionando el aprendizaje de los patrones de idoneidad.
     
La agregación se ha diseñado, además, preservando la información relevante contenida en las imputaciones individuales. Más allá de la suma de los tiempos, se conservan atributos constantes dentro de la fase (como la orden de fabricación o el material) y se derivan variables agregadas de interés analítico, garantizando que la reducción a
un único registro por fase no suponga una pérdida de información útil para el modelado.

**Tiempo bruto y tiempo efectivo: corrección por dedicación**

El MES registra el tiempo de cada imputación como tiempo de reloj (el transcurrido entre inicio y fin), sin considerar la atención real dedicada a la tarea. Como es habitual que un operario gestione varias máquinas a la vez, el reloj sigue corriendo aunque no le dedique el 100 % de su atención.   
Para reflejarlo, el MES registra el porcentaje de dedicación de cada imputación, que se reparte entre las tareas concurrentes.
     
Se definen así dos magnitudes:

- Tiempo bruto (`MinutosReal_total`): el tiempo de reloj sin corregir.
- Tiempo efectivo (`MinutosReal_efectivo`): el tiempo ponderado por la dedicación (tiempo × dedicación / 100), que representa el trabajo realmente invertido en la fase.
       
La corrección se aplica a nivel de imputación, antes de agregar.
     
La distinción es clave para la variable objetivo: el tiempo teórico del ERP asume dedicación plena, por lo que solo es comparable con el tiempo efectivo. Usar el bruto penalizaría injustamente a las asignaciones ejecutadas en multitarea, atribuyendo a la máquina un exceso de tiempo debido a una circunstancia organizativa ajena a su idoneidad. Por ello, el ratio se calcula sobre el tiempo efectivo, conservando el bruto solo para trazabilidad.

In [42]:
# 1. Imputaciones de las fases válidas
imput_validas = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_validas)].copy()

# 2. Corrección por dedicación (tiempo de reloj → tiempo efectivo)
dedicacion = imput_validas["PorcentajeDedicacionNbr"]
imput_validas["MinutosReal_efectivo"] = imput_validas["MinutosRealNbr"] * dedicacion / 100
imput_validas["MinutosPreparacion_efectivo"] = imput_validas["MinutosPreparacionNbr"] * dedicacion / 100
imput_validas["MinutosProduccion_efectivo"] = imput_validas["MinutosProduccionNbr"] * dedicacion / 100

# 3. Máquina dominante de cada fase (la que acumula más tiempo efectivo)
tiempo_fase_nodo = imput_validas.groupby(["SmFaseUid", "smNodoId"])["MinutosReal_efectivo"].sum().reset_index()
idx_dom = tiempo_fase_nodo.groupby("SmFaseUid")["MinutosReal_efectivo"].idxmax()
dominante = tiempo_fase_nodo.loc[idx_dom, ["SmFaseUid", "smNodoId"]].rename(columns={"smNodoId": "MaquinaDominante"})

# 4. Tiempos agregados por fase
tiempos = imput_validas.groupby("SmFaseUid").agg(
    fecha_fase=("FechaCreacionDte", "min"),
    smOrdenId=("smOrdenId", "first"),
    smMaterialId=("smMaterialId", "first"),
    smFaseNme=("smFaseNme", "first"),
    smOrdenNme=("smOrdenNme", "first"),
    # tiempos efectivos (corregidos por dedicación): los del ratio
    MinutosReal_efectivo=("MinutosReal_efectivo", "sum"),
    MinutosPreparacion_efectivo=("MinutosPreparacion_efectivo", "sum"),
    MinutosProduccion_efectivo=("MinutosProduccion_efectivo", "sum"),
    # tiempos brutos (de reloj): solo para trazabilidad
    MinutosReal_total=("MinutosRealNbr", "sum"),
    MinutosPreparacion_total=("MinutosPreparacionNbr", "sum"),
    MinutosProduccion_total=("MinutosProduccionNbr", "sum"),
    dedicacion_media=("PorcentajeDedicacionNbr", "mean"),
    n_imputaciones=("MinutosRealNbr", "size"),
).reset_index()

# 5. Tabla final: una fila por fase
fase_agregada = dominante.merge(tiempos, on="SmFaseUid", how="left")

# 6. Comprobaciones
print(f"Fases en la tabla final: {len(fase_agregada)}")
efectivo_supera = ((fase_agregada["MinutosReal_efectivo"] - fase_agregada["MinutosReal_total"]) > 0.01).sum()
print(f"Fases donde el tiempo efectivo supera al bruto (debe ser 0): {efectivo_supera}")
fase_agregada.head(3)

Fases en la tabla final: 18548
Fases donde el tiempo efectivo supera al bruto (debe ser 0): 0


,SmFaseUid,MaquinaDominante,fecha_fase,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones
0,00013C31-3895-4370-BB3E-4255371E8D8F,TN09,2025-10-02 03:12:05.010,OF2509545,PE1JNCX0012,TPP,Portacápsulas F704501049704(150713),41.76667,41.76667,0.0,41.76667,41.76667,0.0,50.0,2
1,000542EB-1BC1-4F09-8AD8-D892596A91A5,TN10,2025-10-21 12:26:43.433,OF2510311,ELEXA2X0340,TMO,ELECTRODO SUP PERNO 097625004020,41.31667,29.31667,12.0,41.31667,29.31667,12.0,50.0,2
2,0007E37D-C243-4B8B-9372-3E596C45CB70,TN05,2026-01-23 09:40:53.233,OF2600009,BR0XA2X0058,TPG-CNY,Sonderpinole 5658644,12.33334,0.13334,12.2,12.33334,0.13334,12.2,50.0,4


## 5. Fases con el MES sin cerrar

Durante la inspección de los tiempos registrados en el sistema de captura de planta (MES) se observó, en un subconjunto de fases, una incoherencia entre el tiempo real imputado y la suma de sus componentes de preparación y producción. Al analizar estos casos, se identificó que las desviaciones más severas se concentraban en fases con un
número anormalmente elevado de imputaciones. La inspección manual de estas fases reveló el origen del problema: en determinadas ocasiones, una máquina permanecía con la orden abierta en el MES tras finalizar el trabajo efectivo, de modo que el sistema continuaba acumulando imputaciones de tiempo real de forma automática, sin actividad productiva real asociada. Este tiempo inflaba de forma muy acusada el tiempo real de la fase, reflejando un tiempo erróneo.
      
Dado que el tiempo real constituye el numerador de la variable objetivo del estudio (el ratio entre tiempo real y teórico), estos registros habrían distorsionado gravemente el cálculo de la idoneidad, atribuyendo desviaciones elevadas a asignaciones que en realidad fueron correctas y cuyo exceso de tiempo se debía únicamente a un fallo en el cierre del registro. Por este motivo, hablando con el equipo de producción en la planta del estudio, se ha considerado descartar las fases afectadas por este fenómeno. El criterio de identificación combinó la magnitud de la desviación con el número de imputaciones de la fase, lo que permitió aislar los registros defectuosos sin eliminar aquellas fases cuya ligera desviación responde a las condiciones normales de imputación del trabajo.
     
**Criterio aplicado**: un tiempo de preparación o de producción desproporcionado (480 minutos o más en un turno de 8 h) junto con más de 3 imputaciones indica que la máquina quedó abierta en el MES acumulando tiempo sin trabajo real.

In [43]:
UMBRAL_TIEMPO = 480       # minutos (un turno de 8 h) en preparación o en producción
UMBRAL_IMPUTACIONES = 3   # número de imputaciones a partir del cual es sospechoso

fase_agregada["mes_sin_cerrar"] = (
    ((fase_agregada["MinutosPreparacion_total"] >= UMBRAL_TIEMPO)
     | (fase_agregada["MinutosProduccion_total"] >= UMBRAL_TIEMPO)
     | (fase_agregada["MinutosProduccion_total"] + fase_agregada["MinutosPreparacion_total"] >= UMBRAL_TIEMPO))    
    & (fase_agregada["n_imputaciones"] > UMBRAL_IMPUTACIONES)
)

sospechosas = fase_agregada[fase_agregada["mes_sin_cerrar"]]
print(f"Fases marcadas como MES sin cerrar: {len(sospechosas)}\n")
print(sospechosas.nlargest(10, "MinutosProduccion_total")[
    ["SmFaseUid", "MaquinaDominante", "smFaseNme",
     "MinutosPreparacion_total", "MinutosProduccion_total", "n_imputaciones"]
].to_string(index=False))

antes = len(fase_agregada)
fase_agregada = fase_agregada[~fase_agregada["mes_sin_cerrar"]].drop(columns=["mes_sin_cerrar"])
embudo["Fases sin imputaciones mal cerradas"] = len(fase_agregada)
print(f"\nFases antes: {antes} | eliminadas: {antes - len(fase_agregada)} | después: {len(fase_agregada)}")

Fases marcadas como MES sin cerrar: 1254

                           SmFaseUid MaquinaDominante smFaseNme  MinutosPreparacion_total  MinutosProduccion_total  n_imputaciones
5D715E06-67F3-451C-AE5E-982E0083402F             CM04    CV-ALU                4968.85000              32412.53333              77
EDD89008-F84F-427B-802D-24BFEA25DEE4             CM17     MEC03                 585.76666              30921.84999              74
B1378B1E-8601-4B4F-A08C-501D8A0703E6             CM17     MEC03                 585.29999              30917.48332              74
BDD4BE9B-9BCF-4BCF-BCF4-E1BBB9E780DC             CM17     MEC03                 585.71666              30917.48332              74
1A5CBCD3-3A66-4C09-82A2-ADF8A9BE0E3A             CM17     MEC03                2243.95000              28929.40000              74
3A56FC28-C0C7-4219-9701-C91B548BB507             CM17     MEC03                2243.95000              28929.40000              74
68416ED9-521E-405A-8DB1-0D42D2CDDF84     

> Tras esta depuración, el conjunto de análisis queda constituido por 17.294 fases con tiempos fiables.

## 6. Grupo de la máquina dominante

Volviendo a las fases con imputaciones en más de una máquina, se añade a cada fase el grupo de su máquina dominante.

In [44]:
df_fases_MES = fase_agregada.merge(maquina_grupo, left_on="MaquinaDominante", right_on="CodigoMaquina", how="left")
print(f"Fases sin grupo tras la unión (debe ser 0): {df_fases_MES['NombreGrupo'].isna().sum()}")

df_fases_MES = df_fases_MES.drop(columns="CodigoMaquina")  # duplica a MaquinaDominante
print(f"Grupos de máquinas: {df_fases_MES['CodigoGrupo'].nunique()}")
df_fases_MES.head(3)

Fases sin grupo tras la unión (debe ser 0): 0
Grupos de máquinas: 9


,SmFaseUid,MaquinaDominante,fecha_fase,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_efectivo,MinutosPreparacion_efectivo,MinutosProduccion_efectivo,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,dedicacion_media,n_imputaciones,IDMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo
0,00013C31-3895-4370-BB3E-4255371E8D8F,TN09,2025-10-02 03:12:05.010,OF2509545,PE1JNCX0012,TPP,Portacápsulas F704501049704(150713),41.76667,41.76667,0.0,41.76667,41.76667,0.0,50.0,2,0bab3b63-5d31-4de3-85e2-89dc5a3e9bd5,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,27451f15-090c-42a6-82d6-98b7451db3db,TPU,Torns Puma (Taller COURES)
1,000542EB-1BC1-4F09-8AD8-D892596A91A5,TN10,2025-10-21 12:26:43.433,OF2510311,ELEXA2X0340,TMO,ELECTRODO SUP PERNO 097625004020,41.31667,29.31667,12.0,41.31667,29.31667,12.0,50.0,2,93088d93-2c47-4143-8a56-c4f2e86f9290,MORI SEIKI NL2500/1250 Torn C.N.C,6977175d-a2da-4925-a70f-fd0fcc4a2960,TCN,Torns Control Numèric
2,0007E37D-C243-4B8B-9372-3E596C45CB70,TN05,2026-01-23 09:40:53.233,OF2600009,BR0XA2X0058,TPG-CNY,Sonderpinole 5658644,12.33334,0.13334,12.2,12.33334,0.13334,12.2,50.0,4,48e49bcb-1611-4e27-ba11-4cf376cd4a4e,MORI SEIKI NL2500 Torn C.N.C MORI SEIKI NL2500,6977175d-a2da-4925-a70f-fd0fcc4a2960,TCN,Torns Control Numèric


## 7. Guardado

In [45]:
df_fases_MES.to_parquet(PROCESSED / "df_fases_MES.parquet")
guardar_embudo(embudo, "03")

## Resumen

Evolución de las imputaciones (notebooks 01 y 03) y de las fases del MES:

In [46]:
imputaciones = {**cargar_embudo("01"), "Imputaciones en máquinas productivas": embudo["Imputaciones en máquinas productivas"]}
fases = {k: v for k, v in embudo.items() if k.startswith("Fases")}

display(tabla_embudo(imputaciones, "Número de imputaciones"))
display(tabla_embudo(fases, "Número de fases"))

,Número de imputaciones
Imputaciones iniciales,196576
Imputaciones dentro del intervalo temporal,140485
Imputaciones en máquinas productivas,59570


,Número de fases
Fases en máquinas productivas,19149
Fases válidas (dominante >80%),18548
Fases sin imputaciones mal cerradas,17294
